In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("/content/METR-LA.csv")

In [3]:
df.head()

,Unnamed: 0,773869,767541,767542,717447,717446,717445,773062,767620,737529,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
0,2012-03-01 00:00:00,64.375000,67.625000,67.125000,61.500000,66.875000,68.750000,65.125,67.125,59.625000,...,45.625000,65.500,64.500000,66.428571,66.875,59.375000,69.000000,59.250000,69.000000,61.875
1,2012-03-01 00:05:00,62.666667,68.555556,65.444444,62.444444,64.444444,68.111111,65.000,65.000,57.444444,...,50.666667,69.875,66.666667,58.555556,62.000,61.111111,64.444444,55.888889,68.444444,62.875
2,2012-03-01 00:10:00,64.000000,63.750000,60.000000,59.000000,66.500000,66.250000,64.500,64.250,63.875000,...,44.125000,69.000,56.500000,59.250000,68.125,62.500000,65.625000,61.375000,69.857143,62.000
3,2012-03-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000
4,2012-03-01 00:20:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000


In [4]:
df.shape

(34272, 208)

In [5]:
print(df.dtypes[:10])

Unnamed: 0     object
773869        float64
767541        float64
767542        float64
717447        float64
717446        float64
717445        float64
773062        float64
767620        float64
737529        float64
dtype: object


In [6]:
df['Unnamed: 0'] = pd.to_datetime(df['Unnamed: 0'])
df = df.set_index('Unnamed: 0')

In [7]:
df.columns

Index(['773869', '767541', '767542', '717447', '717446', '717445', '773062',
       '767620', '737529', '717816',
       ...
       '772167', '769372', '774204', '769806', '717590', '717592', '717595',
       '772168', '718141', '769373'],
      dtype='object', length=207)

In [8]:
print(type(df.index)) # checking th data type for columns

<class 'pandas.core.indexes.datetimes.DatetimeIndex'>


In [9]:
print(df.index[1] - df.index[0])  #sanuity chexkk
print(df.index.to_series().diff().value_counts().head())

0 days 00:05:00
Unnamed: 0
0 days 00:05:00    34271
Name: count, dtype: int64


In [10]:
# range check

print(df.index.min())
print(df.index.max())

2012-03-01 00:00:00
2012-06-27 23:55:00


Feature enginnering

In [11]:
speed_data = df.values
print(speed_data.shape)

(34272, 207)


In [35]:
# time features
df["hour"] = df.index.hour
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

df["day"] = df.index.dayofweek
df["day_sin"] = np.sin(2*np.pi*df["day"]/7)
df["day_cos"] = np.cos(2*np.pi*df["day"]/7)

df["is_weekend"] = (df["day"] >= 5).astype(int)

time_features = df[['hour_sin','hour_cos','day_sin','day_cos','is_weekend']].values

print("Speed data shape:", speed_data.shape)


Speed data shape: (34272, 207)


In [37]:
# 3. Train / Val / Test Split

T = len(df)

train_end = int(T * 0.7)
val_end = int(T * 0.8)

speed_train = speed_data[:train_end]
speed_val = speed_data[train_end:val_end]
speed_test = speed_data[val_end:]

time_train = time_features[:train_end]
time_val = time_features[train_end:val_end]
time_test = time_features[val_end:]


In [39]:
# 4. Normalize Speed

mean = speed_train.mean()
std = speed_train.std()

speed_train = (speed_train - mean) / std
speed_val = (speed_val - mean) / std
speed_test = (speed_test - mean) / std

# save scaler for inference
with open("/content/scaler.pkl", "wb") as f:
    pickle.dump({"mean": mean, "std": std}, f)


In [40]:
# 5. Expand Time Features

num_nodes = speed_train.shape[1]

def expand_time(time_data):
    return np.repeat(time_data[:, np.newaxis, :], num_nodes, axis=1)

time_train_exp = expand_time(time_train)
time_val_exp = expand_time(time_val)
time_test_exp = expand_time(time_test)


In [41]:
# 6. Add Speed Feature Dimension

speed_train = speed_train[..., np.newaxis]
speed_val = speed_val[..., np.newaxis]
speed_test = speed_test[..., np.newaxis]


In [42]:
# 7. Combine Features

train_full = np.concatenate([speed_train, time_train_exp], axis=-1)
val_full = np.concatenate([speed_val, time_val_exp], axis=-1)
test_full = np.concatenate([speed_test, time_test_exp], axis=-1)

print("Train full shape:", train_full.shape)
print("Val full shape:", val_full.shape)
print("Test full shape:", test_full.shape)


Train full shape: (23990, 207, 6)
Val full shape: (3427, 207, 6)
Test full shape: (6855, 207, 6)


In [43]:
# 8. Sliding Window Generator


def create_sequences_full(data, input_len=12, output_len=6):

    X = []
    Y = []

    for i in range(len(data) - input_len - output_len):

        X.append(data[i:i+input_len])

        # only speed is predicted
        Y.append(data[i+input_len:i+input_len+output_len,:,0])

    return np.array(X), np.array(Y)


X_train, Y_train = create_sequences_full(train_full)
X_val, Y_val = create_sequences_full(val_full)
X_test, Y_test = create_sequences_full(test_full)

print("Train:", X_train.shape, Y_train.shape)


Train: (23972, 12, 207, 6) (23972, 6, 207)


In [44]:
# 9. Save Arrays

np.save("/content/X_train.npy", X_train)
np.save("/content/Y_train.npy", Y_train)

np.save("/content/X_val.npy", X_val)
np.save("/content/Y_val.npy", Y_val)

np.save("/content/X_test.npy", X_test)
np.save("/content/Y_test.npy", Y_test)

print("Saved training datasets successfully")


Saved training datasets successfully


In [45]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [46]:
import os
os.listdir('/content/drive/MyDrive')

['Colab Notebooks',
 'git and git hub.pdf',
 'data science methodoiology.pdf',
 'ec council.pdf',
 'database and sql.pdf',
 'data anlaysis with python.pdf',
 'data vissulation with python.pdf',
 'DSA - Real World Examples.pdf',
 'DSA PATTERNS.pdf',
 'IMPORTANT DSA cheatsheet.pdf',
 'Document 25_removed.pdf',
 'pneumonia_model.h5',
 'lda_model_1.pkl',
 'vectorizer_1.pkl',
 'QA_Project',
 'Euducation_Loan.ipynb']

In [47]:
!mkdir "/content/drive/MyDrive/STGCN_Project"

In [48]:
import os
os.listdir('/content/drive/MyDrive')

['Colab Notebooks',
 'git and git hub.pdf',
 'data science methodoiology.pdf',
 'ec council.pdf',
 'database and sql.pdf',
 'data anlaysis with python.pdf',
 'data vissulation with python.pdf',
 'DSA - Real World Examples.pdf',
 'DSA PATTERNS.pdf',
 'IMPORTANT DSA cheatsheet.pdf',
 'Document 25_removed.pdf',
 'pneumonia_model.h5',
 'lda_model_1.pkl',
 'vectorizer_1.pkl',
 'QA_Project',
 'Euducation_Loan.ipynb',
 'STGCN_Project']

In [49]:
!mv *.npy /content/drive/MyDrive/STGCN_Project/
!mv *.pkl /content/drive/MyDrive/STGCN_Project/
!mv *.pth /content/drive/MyDrive/STGCN_Project/

mv: cannot stat '*.pth': No such file or directory


In [50]:
os.listdir('/content/drive/MyDrive/STGCN_Project')

['X_test.npy',
 'X_train.npy',
 'X_val.npy',
 'Y_test.npy',
 'Y_train.npy',
 'Y_val.npy',
 'scaler.pkl']